# Aula 6 — Aprendizado Supervisionado: Classificação
## *Ensinando o computador a separar as coisas em categorias*

---

> **Data:** *(preencher)*  
> **Professor(a):** *(preencher)*

## Aquecimento: isso é um carro?

Na última aula mostramos uma sequência de imagens e perguntamos: **isso é um carro?**

Tinha carro novo, carro velho, carro de brinquedo, carro capotado, carro sem rodas, moto, carrinho de plástico...

E você, sem pensar muito, respondeu a cada uma — **sim, não, sim, hm depende, não...**

Agora a pergunta que importa: **como você fez isso?**

Você não consultou nenhum manual. Não calculou nada. Usou um padrão que internalizou ao longo da vida, vendo milhares de exemplos de carros e não-carros.

**Classificação em Machine Learning é exatamente isso — só que para o computador.**

---

## 1. Classificação vs Regressão — qual a diferença?

Nas últimas aulas, trabalhamos com **Regressão**: prever um número contínuo.

- *"Quanto custa esse diamante?"* → $4.250
- *"Qual a temperatura amanhã?"* → 28°C
- *"Qual será o salário desse candidato?"* → R$ 8.400

**Classificação** é diferente: você quer prever uma **categoria** — uma das opções de uma lista fechada.

- *"Esse email é spam ou não spam?"* → spam / não spam
- *"Esse tumor é maligno ou benigno?"* → maligno / benigno
- *"Esse cliente vai cancelar o plano?"* → sim / não
- *"Qual fruta é essa?"* → maçã / banana / uva / laranja

### A diferença fundamental:

| | Regressão | Classificação |
|---|---|---|
| **O que prevê** | Um número | Uma categoria |
| **Exemplo de saída** | $4.250 | "spam" |
| **Métrica principal** | MAE, RMSE, R² | Acurácia, Precisão, Recall |
| **Modelos comuns** | Regressão Linear | Regressão Logística, Árvore de Decisão |

### Binária vs Multiclasse

Quando a classificação tem **apenas 2 categorias** (sim/não, spam/legítimo, fraude/não fraude), chamamos de **classificação binária**.

Quando tem **3 ou mais categorias** (maçã/banana/uva, A/B/C, tipo1/tipo2/tipo3), chamamos de **classificação multiclasse**.

Vamos começar pelo caso mais simples: binária.

---

## 2. O problema de hoje: prever se um paciente tem diabetes

Vamos trabalhar com o **Pima Indians Diabetes Dataset** — um conjunto de dados médicos muito usado para aprender classificação.

**O problema:** com base em informações clínicas de uma paciente, prever se ela tem diabetes ou não.

### Variáveis do dataset:

| Coluna | O que é |
|---|---|
| `Pregnancies` | Número de gestações |
| `Glucose` | Nível de glicose no sangue |
| `BloodPressure` | Pressão arterial (mm Hg) |
| `SkinThickness` | Espessura da pele (mm) |
| `Insulin` | Nível de insulina (mu U/ml) |
| `BMI` | Índice de Massa Corporal |
| `DiabetesPedigreeFunction` | Histórico familiar de diabetes |
| `Age` | Idade |
| `Outcome` | **0 = não tem diabetes / 1 = tem diabetes** ← nosso alvo |

**Por que esse dataset?** Porque é um problema real, com impacto claro, e qualquer pessoa consegue entender o que está sendo previsto — sem precisar saber nada de joalheria ou finanças.

Vamos carregar os dados:

In [ ]:
import shutil
import urllib.request
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Na primeira vez a base é baixada para data_raw/ (o original, que fica de
# referência) e copiada para data/ (a cópia de trabalho, que é a que vamos usar).
# Bagunçou a base? Apague data/diabetes.csv e rode esta célula de novo.
url = 'https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv'
Path('data_raw').mkdir(exist_ok=True)
Path('data').mkdir(exist_ok=True)

if not Path('data_raw/diabetes.csv').exists():
    print('Baixando diabetes.csv...')
    urllib.request.urlretrieve(url, 'data_raw/diabetes.csv')
if not Path('data/diabetes.csv').exists():
    shutil.copy('data_raw/diabetes.csv', 'data/diabetes.csv')

df = pd.read_csv('data/diabetes.csv')

print(f"Dataset carregado: {df.shape[0]} pacientes, {df.shape[1]} colunas")
df.head()

In [ ]:
# Verificando tipos e valores ausentes


In [ ]:
# Estatísticas básicas


### Explorando o alvo: quantos têm diabetes?

Antes de qualquer modelo, precisamos entender a distribuição das classes. Isso é crucial — e logo você vai entender por quê.

In [ ]:
# Gráfico de barras

# Glicose por classe (pré-visualização da separação)


**Observações importantes:**

1. As classes estão **desbalanceadas**: 65% sem diabetes, 35% com. Isso não é dramático, mas precisamos lembrar — acurácia sozinha pode enganar aqui.

2. O histograma de glicose já mostra uma **separação visual** entre as duas classes. Pessoas com diabetes tendem a ter glicose mais alta. Isso sugere que glicose será uma variável importante para o modelo.

Vamos explorar mais:

In [ ]:
# Comparando médias das variáveis entre as duas classes


In [ ]:
# Boxplots das variáveis mais relevantes


Os boxplots confirmam: glicose, BMI e idade são maiores em média nos pacientes com diabetes. O modelo vai aprender a usar essas diferenças para classificar.

---

## 3. Pré-processamento

Antes de treinar qualquer modelo, precisamos preparar os dados. O processo é o mesmo da aula anterior:

1. Separar features (X) e alvo (y)
2. Dividir em treino e teste
3. Aplicar scaling nas features

In [ ]:
# Separando features e alvo

# Dividindo treino/teste

# Scaling


**O parâmetro `stratify=y`:** quando as classes estão desbalanceadas, existe o risco de a divisão aleatória colocar todos os casos raros em um único lado. O `stratify` garante que a proporção de cada classe seja **preservada** no treino e no teste. Sempre use isso em problemas de classificação com desbalanceamento.

---

## 4. Primeiro modelo: Regressão Logística

### O nome confunde — não é regressão!

Sim, o nome tem "Regressão". Não, ela não prevê números. É um classificador. O nome é histórico e todos precisamos conviver com isso.

### O que ela faz?

A Regressão Logística pega as variáveis de entrada, calcula uma combinação delas (igual à regressão linear), e depois **transforma esse número em uma probabilidade** entre 0 e 1.

```
Passo 1 (igual à regressão linear):
z = a₁×glicose + a₂×bmi + a₃×idade + ... + b

Passo 2 (o que é diferente — a função logística):
probabilidade = 1 / (1 + e^(-z))
```

Esse segundo passo — a **função logística** (ou sigmóide) — é o truque. Ela espreme qualquer número real para o intervalo [0, 1]. Então o resultado é sempre uma probabilidade.

### Da probabilidade para a classe

Com a probabilidade em mãos, aplicamos um **limiar de decisão**:

```
se probabilidade >= 0.5 → prevê classe 1 (tem diabetes)
se probabilidade  < 0.5 → prevê classe 0 (não tem diabetes)
```

O limiar padrão é 0.5, mas pode ser ajustado. Voltaremos a esse ponto na Aula 7, quando falarmos de Precisão vs Recall.

### A fronteira de decisão

A Regressão Logística cria uma **linha reta** (ou hiperplano, em múltiplas dimensões) que separa as duas classes. Tudo de um lado = classe 0. Tudo do outro = classe 1.

Isso a torna simples, interpretável e rápida — mas também limita: se as classes não forem separáveis por uma linha reta, ela vai ter dificuldade.

In [ ]:
# Visualizando a função sigmóide — o coração da Regressão Logística


In [ ]:
# Treinando o modelo

# Previsões


In [ ]:
# Matriz de Confusão

# Matriz

# Distribuição das probabilidades previstas


### Lendo os resultados:

**Matriz de Confusão:** ela cruza o que o modelo previu com o que de fato aconteceu, separando os acertos e os erros em VP, VN, FP e FN. No contexto médico:
- **Falso Negativo** (disse "sem diabetes", mas o paciente tem) → o mais perigoso. O paciente não receberá tratamento.
- **Falso Positivo** (disse "tem diabetes", mas o paciente não tem) → preocupante, mas menos grave — levará a exames adicionais.

**Distribuição das probabilidades:** quanto mais separadas as duas montanhas, melhor o modelo distingue as classes. Se estiverem sobrepostas, o modelo tem dificuldade.

---

### Os coeficientes da Regressão Logística

Assim como na Regressão Linear, cada variável tem um peso. Aqui, o coeficiente positivo significa que a variável **aumenta a probabilidade** de diabetes. Negativo significa que **diminui**.

In [ ]:
# Seu código aqui


---

## 5. Segundo modelo: Árvore de Decisão

### A intuição — como você mesmo classifica

Imagine que você é médico e precisa decidir se um paciente tem diabetes. Você provavelmente faria algo assim:

```
Glicose > 140?
├── SIM → BMI > 30?
│         ├── SIM → DIABETES (alta probabilidade)
│         └── NÃO → Verificar outros fatores...
└── NÃO → Idade > 50?
           ├── SIM → Verificar insulina...
           └── NÃO → SEM DIABETES (baixa probabilidade)
```

Isso é exatamente o que uma **Árvore de Decisão** faz: ela aprende uma sequência de perguntas (condições) que, respondidas em ordem, levam a uma classificação.

### Como a árvore aprende as perguntas?

O algoritmo testa, para cada variável, **todos os possíveis pontos de corte** e escolhe aquele que melhor **separa as classes**.

**Critério de separação — Gini Impurity:**

Imagine que você tem uma caixa com bolas vermelhas e azuis completamente misturadas. Se você pegar uma bola aleatoriamente, qual a chance de pegar a errada? Alta — a caixa está "impura".

Agora imagine que todas as bolas vermelhas estão de um lado e todas as azuis do outro. Se pegar uma bola de qualquer lado, você sabe o que vai pegar — a caixa está "pura".

O Gini mede exatamente essa impureza. A árvore escolhe os cortes que **minimizam o Gini** — que deixam cada lado o mais puro possível.

### Vantagens da Árvore de Decisão:
- ✅ **Interpretável:** você consegue visualizar e explicar exatamente por que o modelo tomou cada decisão
- ✅ **Não precisa de scaling:** trabalha bem com variáveis em escalas diferentes
- ✅ **Captura relações não-lineares:** não precisa que a separação seja uma linha reta
- ✅ **Funciona com variáveis categóricas:** sem precisar de encoding

### Desvantagem principal:
- ⚠️ **Muito suscetível a overfitting:** uma árvore sem limite de profundidade memoriza o treino perfeitamente — mas falha no teste.

In [ ]:
# Primeiro, vamos ver o que acontece sem limitar a profundidade

# Agora limitando a profundidade


**Olha o overfitting!** A árvore sem limite acerta **100% no treino** (decorou tudo!) mas vai mal no teste. Com `max_depth=4`, o desempenho no treino cai um pouco, mas o teste melhora — o modelo generalizou melhor.

Agora vamos visualizar a árvore — isso é uma das coisas mais poderosas desse modelo:

In [ ]:
# Visualizando a árvore de decisão


### Como ler a árvore:

Cada **nó** (caixinha) mostra:
- A **pergunta** que está sendo feita (ex: `Glucose <= 0.53`)
- O **Gini** — impureza desse nó (0 = puro, 0.5 = máxima mistura)
- O **samples** — quantos exemplos de treino chegaram aqui
- O **value** — quantos de cada classe: [sem diabetes, com diabetes]
- A **class** — qual seria a previsão se o caminho parasse aqui

A **cor** indica a classe dominante: azul = sem diabetes, laranja = com diabetes. Mais escuro = mais certeza.

Para classificar um paciente novo, você segue o caminho da raiz até uma folha respondendo as perguntas.

> **Atenção:** os valores das perguntas estão em escala padronizada (pós-StandardScaler). Por isso aparece `Glucose <= 0.53` em vez de `Glucose <= 140`. O modelo funciona correto — é só uma questão de como os dados foram transformados.

In [ ]:
# Matriz de confusão da árvore

# Importância das features


**Feature Importance (Importância das Variáveis):** a árvore de decisão nos dá de graça uma medida de quais variáveis mais contribuíram para as divisões. Glicose domina — faz sentido médico!

---

## 6. Comparando os dois modelos

Agora temos dois classificadores. Qual é melhor?

Como sempre em análise de dados: não existe resposta sem contexto. Vamos comparar por múltiplos ângulos.

In [ ]:
# Cross-validation para os dois modelos

# Tabela comparativa


In [ ]:
# Visualização comparativa


### Quem ganhou?

No caso específico desse dataset, os modelos têm desempenhos parecidos. Mas cada um tem características diferentes:

| | Regressão Logística | Árvore de Decisão |
|---|---|---|
| **Interpretabilidade** | Coeficientes lineares | Regras de decisão visuais |
| **Fronteira de decisão** | Linha reta | Linhas paralelas aos eixos |
| **Risco de overfitting** | Baixo | Alto (sem `max_depth`) |
| **Precisa de scaling** | Sim | Não |
| **Velocidade** | Muito rápida | Rápida |
| **Quando usar** | Relações lineares, quando interpretabilidade importa | Relações não-lineares, quando precisa explicar as regras |

---

## 7. Fronteiras de Decisão — visualizando como cada modelo "vê" o problema

Para entender intuitivamente a diferença entre os dois modelos, vamos plotar as **fronteiras de decisão** — a linha que separa os dois lados para cada modelo.

Para poder visualizar em 2D, vamos usar apenas as duas variáveis mais importantes: **Glicose** e **BMI**.

In [ ]:
# Usando apenas 2 features para poder visualizar em 2D

# Treina os dois modelos nas 2 features


**O que você vê:**

- **Regressão Logística:** fronteira **diagonal e suave** — uma única linha reta divide o espaço em dois. Simples e elegante, mas pode não capturar padrões complexos.

- **Árvore de Decisão:** fronteira **em degraus** — só faz cortes horizontais e verticais (paralelos aos eixos). Cada degrau corresponde a uma pergunta da árvore. Mais flexível, mas pode criar formas estranhas.

Nenhuma captura perfeitamente a separação (há muita sobreposição) — o que confirma que diabetes é difícil de prever só com glicose e BMI.

---

## 8. Como um modelo classifica um paciente novo?

Vamos fechar com um exemplo prático: criar um paciente hipotético e ver o que os dois modelos dizem.

In [ ]:
# Paciente hipotético (valores nas unidades originais)

# Aplicando o mesmo scaling

# Previsões


> **Curiosidade:** esse paciente é exatamente o primeiro registro do dataset original, e a classe real é **1 (tem diabetes)**. Veja se os dois modelos acertaram!

---

## Recapitulando o que aprendemos hoje

```
CLASSIFICAÇÃO
│
├── O que é?
│   Prever uma CATEGORIA, não um número.
│   Binária (2 classes) ou Multiclasse (3+)
│
├── Regressão Logística
│   ├── Calcula combinação linear das features
│   ├── Aplica função sigmóide → transforma em probabilidade [0,1]
│   ├── Aplica limiar (padrão: 0.5) → classe final
│   ├── Fronteira de decisão: linha reta
│   └── Boa interpretabilidade via coeficientes
│
├── Árvore de Decisão
│   ├── Aprende sequência de perguntas (if/else)
│   ├── Cada divisão minimiza impureza (Gini)
│   ├── Fronteira de decisão: degraus (paralelos aos eixos)
│   ├── Muito visual e interpretável
│   └── Risco alto de overfitting — controlar max_depth!
│
└── Como avaliar?
    (usamos hoje; aprofundamos na Aula 7)
    ├── Acurácia (mas cuidado com desbalanceamento)
    ├── Precisão, Recall, F1
    ├── Matriz de Confusão
    └── Cross-Validation
```

---

## Exercícios

---

### 🟢 Fácil

**Exercício 1 — Entendendo o limiar**

O modelo de Regressão Logística usa limiar 0.5 por padrão.

Mude o limiar para **0.3** e recalcule a Matriz de Confusão e as métricas.
- Quantos Falsos Negativos havia antes? E agora?
- O Recall aumentou ou diminuiu?
- No contexto médico (detectar diabetes), qual limiar você preferiria?

*Dica:* use `y_proba_lr >= 0.3` para gerar as previsões com novo limiar.

In [ ]:
# Exercício 1


**Exercício 2 — Árvore mais funda ou mais rasa?**

Treine Árvores de Decisão com `max_depth` = 1, 2, 3, 4, 5, 10 e None (sem limite).

Para cada uma, calcule acurácia no treino e no teste.

Plote os dois em um gráfico de linha com o `max_depth` no eixo X.

- Qual profundidade parece o melhor equilíbrio?
- A partir de qual profundidade o overfitting fica evidente?

In [ ]:
# Exercício 2


---

### 🟡 Médio

**Exercício 3 — Feature engineering simples**

Experimente treinar os modelos usando **apenas as 3 variáveis mais importantes** segundo a Árvore de Decisão.

- O desempenho cai muito?
- Quais variáveis foram descartadas?
- Um modelo mais simples (menos features) pode ter alguma vantagem?

In [ ]:
# Exercício 3


**Exercício 4 — Criando um paciente e interpretando**

Crie 3 pacientes fictícios:
1. Perfil de baixo risco (glicose baixa, BMI normal, jovem)
2. Perfil de médio risco
3. Perfil de alto risco (glicose alta, BMI elevado, histórico familiar)

Para cada um:
- Calcule a probabilidade prevista pelos dois modelos
- Mostre qual caminho o paciente percorre na árvore de decisão

*Dica para o caminho na árvore:* use `arvore.decision_path(paciente_sc)`

In [ ]:
# Exercício 4


---

### 🔴 Difícil

**Exercício 5 — O custo assimétrico dos erros**

No contexto médico, um **Falso Negativo** (dizer que o paciente não tem diabetes quando tem) é muito mais grave do que um Falso Positivo.

Imagine que o custo de um FN é **5x maior** do que o custo de um FP.

1. Para diferentes limiares (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7), calcule o **custo total** = `FP × 1 + FN × 5`
2. Qual limiar minimiza esse custo?
3. Plote o gráfico de custo total por limiar
4. Reflita: por que usar a acurácia como métrica principal seria errado nesse problema?

In [ ]:
# Exercício 5


**Exercício 6 — Multiclasse (bônus)**

Até agora trabalhamos com classificação binária (diabetes: sim/não).

Carregue o dataset `iris` do seaborn (`sns.load_dataset('iris')`) — que tem 3 classes de flores — e aplique o pipeline completo:

1. Exploração das classes
2. Treino/teste com `stratify`
3. Regressão Logística e Árvore de Decisão
4. Matriz de confusão (agora é 3×3!)
5. Comparação com cross-validation

O que muda quando passamos de 2 para 3 classes? A Matriz de Confusão muda como?

In [ ]:
# Exercício 6


---

## Resumo Final

| Conceito | O que é | Como usar |
|---|---|---|
| **Classificação** | Prever uma categoria | Quando o alvo é sim/não, A/B/C |
| **Binária vs Multiclasse** | 2 classes vs 3+ classes | Define a complexidade do problema |
| **`stratify=y`** | Preserva proporção de classes no split | Sempre usar com classes desbalanceadas |
| **Regressão Logística** | Usa sigmóide para prever probabilidade | Rápida, interpretável, fronteira linear |
| **Função Sigmóide** | Transforma qualquer número em [0,1] | É o coração da Reg. Logística |
| **Limiar de decisão** | Ponto de corte para classificar | Padrão 0.5, mas ajustável conforme o custo dos erros |
| **Árvore de Decisão** | Sequência de perguntas if/else | Interpretável, não precisa de scaling |
| **Gini Impurity** | Mede impureza de um nó | Árvore minimiza o Gini em cada divisão |
| **`max_depth`** | Profundidade máxima da árvore | Controla overfitting — sempre defina! |
| **Feature Importance** | Contribuição de cada variável | Gratuito na Árvore de Decisão |
| **Fronteira de Decisão** | Como o modelo separa as classes | Linear (LR) vs degraus (Árvore) |

> **A frase que resume a aula:** *Classificação é regressão com a saída sendo uma categoria. Os modelos são diferentes, mas o pipeline — explorar, pré-processar, treinar, avaliar — é sempre o mesmo.*